### Mounting Google Drive

In [60]:
from google.colab import drive

In [61]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Importing Required Libaries

In [62]:
import re
import spacy
import pandas as pd

### Loading the dataset

In [63]:
df = pd.read_csv("/content/drive/MyDrive/cleaned/cleaned_products.csv")

### Quick exploration

In [64]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 133 entries, 0 to 132
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   product_id   133 non-null    object 
 1   name         133 non-null    object 
 2   description  133 non-null    object 
 3   rating       133 non-null    float64
dtypes: float64(1), object(3)
memory usage: 4.3+ KB


In [65]:
df.head()

,product_id,name,description,rating
0,381863621,ORIGINAL Condom Boxers For Men- 3pcs,These pair of boxers give good fit and sits ap...,3.5
1,401669634,Ladies Sexy Lace Panties - Set Of 6 M/L HIPS,"THIS PANTIES IS FOR DOUBLE HIPS ONLY , A real ...",3.4
2,388971592,"5-piece Men's Watch And Bracelet Set, Watch Se...","For more exciting products, please follow my s...",3.6
3,50091396,Curren Exotic Leather Strap Military Sports Me...,Nothing beats the timeless appeal of an elegan...,3.7
4,94668946,TOUPHY T-Shirts +Shorts 100% Cotton Men's Shor...,"Whether formally dressed or in casuals, style ...",3.4


### Text Preprocessing

The reason I didn’t use traditional NLP text preprocessing (where stopwords, digits, and punctuation are removed, and text stemming is applied) is that useful signals would be lost in the process.

In [66]:
nlp = spacy.load("en_core_web_sm")

In [86]:
def preprocess_description(text):
  """Cleans product descriptions by standardizing whitespace, fixing formatting, and removing duplicate sentences."""

  # normalise white space
  text = re.sub(r'\s+', ' ', text)

  # Inserts a period when a lowercase letter is immediately followed by an uppercase letter
  text = re.sub(r'([a-z])([A-Z])', r'\1. \2', text)

  # Replaces the Unicode non-breaking space with a normal space
  text = text.replace('\xa0', ' ')

  seen = set()
  unique = []

  doc = nlp(text)
  for sent in doc.sents:
      normalized = sent.text.strip().lower()
      if not normalized:
          continue
      if normalized not in seen:
          seen.add(normalized)
          unique.append(sent.text.strip())

  return " ".join(unique)

In [87]:
df['clean_description'] = df['description'].apply(preprocess_description)

### Checking descriptions vs cleaned descriptions

In [88]:
from rich.console import Console
from rich.panel import Panel

def display_text(text):
  console = Console()
  console.print(Panel(text, title="Long Text", expand=False))

In [89]:
display_text(df.loc[10, "description"])

╭─────────────────────────────────────────────────── Long Text ───────────────────────────────────────────────────╮
│ ‎Meet the Valenzo Matte Unisex Watch a sleek, versatile timepiece designed for those who appreciate effortless   │
│ style with a bold edge. Featuring a matte finish, clean dial design, and a gender-neutral silhouette, this      │
│ watch redefines modern minimalism, Perfect for any wrist, any outfit, any moment—Valenzo is the quiet           │
│ confidence that sets you apart,Mens Sophisticated Super Duper Display Good Looking Trendy Watch. A perfect mesh │
│ Chain Wristwatch Wristwatch ideal gift for your loved ones as This Watches are suitable for all                 │
│ occasions.Elegant and fashionable design satisfies your costume matching & A good present for your families,    │
│ friends, Analog timepieces have not only upgraded their high-tech features but have also caught up to the       │
│ design challenge. With a Analog watch, you get the best of both worlds - smart, sophisticated shape and         │
│ state-of-the-art technology. The modern Analog watch is fast shaping up to be the accessory of the future,      │
│ rather than a novelty of a bygone era. Dressed up for a business lunch or paired down for a hike in the         │
│ mountains, the Analog watch wears as well as it performs. Important information DAILY WEAR ANALOG,It has a      │
│ buckle clasp type which overall complements the overall look of the watch. The buckle clasp helps the watch to  │
│ fit comfortably on your wrist. Stylish The look. ‎Mens Sophisticated Super Duper Display Good Looking Trendy     │
│ Watch. A perfect mesh Chain Wristwatch Wristwatch ideal gift for your loved ones as This Watches are suitable   │
│ for all occasions.Elegant and fashionable design satisfies your costume matching & A good present for your      │
│ families, friends, Analog timepieces have not only upgraded their high-tech features but have also caught up to │
│ the design challenge. With a Analog watch, you get the best of both worlds - smart, sophisticated shape and     │
│ state-of-the-art technology. The modern Analog watch is fast shaping up to be the accessory of the future,      │
│ rather than a novelty of a bygone era. Dressed up for a business lunch or paired down for a hike in the         │
│ mountains, the Analog watch wears as well as it performs. Important information DAILY WEAR ANALOG,It has a      │
│ buckle clasp type which overall complements the overall look of the watch. The buckle clasp helps the watch to  │
│ fit comfortably on your wrist. Stylish The look.                                                                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [90]:
display_text(df.loc[10, "clean_description"])

╭─────────────────────────────────────────────────── Long Text ───────────────────────────────────────────────────╮
│ ‎Meet the Valenzo Matte Unisex Watch a sleek, versatile timepiece designed for those who appreciate effortless   │
│ style with a bold edge. Featuring a matte finish, clean dial design, and a gender-neutral silhouette, this      │
│ watch redefines modern minimalism, Perfect for any wrist, any outfit, any moment—Valenzo is the quiet           │
│ confidence that sets you apart,Mens Sophisticated Super Duper Display Good Looking Trendy Watch. A perfect mesh │
│ Chain Wristwatch Wristwatch ideal gift for your loved ones as This Watches are suitable for all occasions.      │
│ Elegant and fashionable design satisfies your costume matching & A good present for your families, friends,     │
│ Analog timepieces have not only upgraded their high-tech features but have also caught up to the design         │
│ challenge. With a Analog watch, you get the best of both worlds - smart, sophisticated shape and                │
│ state-of-the-art technology. The modern Analog watch is fast shaping up to be the accessory of the future,      │
│ rather than a novelty of a bygone era. Dressed up for a business lunch or paired down for a hike in the         │
│ mountains, the Analog watch wears as well as it performs. Important information DAILY WEAR ANALOG,It has a      │
│ buckle clasp type which overall complements the overall look of the watch. The buckle clasp helps the watch to  │
│ fit comfortably on your wrist. Stylish The look. ‎Mens Sophisticated Super Duper Display Good Looking Trendy     │
│ Watch.                                                                                                          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

### Saving the final dataset

In [96]:
from google.colab import files

In [97]:
df.to_csv("processed_products.csv")

In [98]:
files.download("processed_products.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>